# Extract LME Control Variables

Builds `controls.csv` — one row per `stim_id` — containing the lexical control
variables needed for linear mixed-effects models predicting N400 amplitude:

| Column | Description | Source |
|---|---|---|
| `word_length` | Character length of critical word | computed |
| `word_position` | 0-based word index in sentence | computed |
| `zipf_freq` | Zipf frequency (log10/billion) | wordfreq |
| `log10_word_freq` | log10 of raw frequency probability | wordfreq |
| `coltheart_n` | Orthographic neighborhood size | NLTK words |

Note: `n_tokens` (BPE subword count) is already in the model output CSVs from `pipeline.ipynb`.

Merge key: `stim_id` (globally unique across all three datasets).

In [2]:
import re
import numpy as np
import pandas as pd
from collections import defaultdict
from functools import lru_cache

from wordfreq import word_frequency, zipf_frequency
import nltk

In [3]:
STIMS_PATH   = 'stims_for_modeling.csv'
OUTPUTS_DIR  = 'outputs/'
CONTROLS_OUT = 'controls.csv'

## 1. Load stimuli

In [4]:
stims = pd.read_csv(STIMS_PATH)
print(f"Loaded {len(stims)} stimuli across {stims['dataset'].nunique()} datasets")
stims.head()

Loaded 2808 stimuli across 3 datasets


,dataset,stim_id,stim,stim_lower,critical_word
0,frank_2015,1140,I cannot tell you more,i cannot tell you more,cannot
1,frank_2015,1141,I cannot tell you more,i cannot tell you more,tell
2,frank_2015,1142,I cannot tell you more,i cannot tell you more,you
3,frank_2015,1143,I cannot tell you more,i cannot tell you more,more
4,frank_2015,1144,Helen ran to the toilet,helen ran to the toilet,ran


## 2. Basic controls: word length and sentence position

**Word position** uses regex tokenization (`\b\w+\b`) rather than `.split()` so that
punctuation attached to tokens (e.g. `"wondered,"`) doesn't cause a mismatch with
the clean critical word string.  Position is 0-based (first word = 0).

In [5]:
def get_word_position(sentence: str, critical_word: str) -> int:
    """0-based index of critical_word in the tokenized sentence."""
    tokens = re.findall(r'\b\w+\b', sentence.lower())
    cw = critical_word.lower()
    try:
        return tokens.index(cw)
    except ValueError:
        # Contraction mismatch: sentence stores 'dont', critical_word is "don't"
        cw_no_apos = cw.replace("'", "")
        try:
            return tokens.index(cw_no_apos)
        except ValueError:
            return -1  # should not happen given clean data


stims['word_length']   = stims['critical_word'].str.len()
stims['word_position'] = stims.apply(
    lambda r: get_word_position(r['stim_lower'], r['critical_word']), axis=1
)

# Sanity check: any failures?
n_fail = (stims['word_position'] == -1).sum()
print(f"Unmatched positions: {n_fail} / {len(stims)}")
stims[['stim_id', 'critical_word', 'word_length', 'word_position']].head(8)

Unmatched positions: 0 / 2808


,stim_id,critical_word,word_length,word_position
0,1140,cannot,6,1
1,1141,tell,4,2
2,1142,you,3,3
3,1143,more,4,4
4,1144,ran,3,1
5,1145,the,3,3
6,1146,toilet,6,4
7,1147,to,2,2


## 3. Word frequency (wordfreq)

`zipf_frequency` returns the Zipf scale (log10 of occurrences per billion tokens).
This ranges ~0–7 and is 0 for words below 1-per-billion — rare but handled gracefully.

`log10_word_freq` is the raw log10 of the frequency probability (large negative values
for rare words).  Both are kept; use whichever your LME framework convention prefers.

Both metrics are computed using the `large` wordfreq corpus (Parallel Bible Corpus +
OpenSubtitles + OSCAR, ~2B English tokens), consistent with Michaelov et al. (2024)
who used the `wordfreq` package.

In [6]:
stims['zipf_freq'] = stims['critical_word'].apply(
    lambda w: zipf_frequency(w.lower(), 'en')
)

stims['log10_word_freq'] = stims['critical_word'].apply(
    lambda w: np.log10(word_frequency(w.lower(), 'en') + 1e-9)
)

stims[['critical_word', 'zipf_freq', 'log10_word_freq']].describe()

,zipf_freq,log10_word_freq
count,2808.000000,2808.000000
mean,5.004103,-3.995636
std,1.502235,1.501694
min,0.000000,-9.000000
25%,3.940000,-5.059932
50%,4.920000,-4.079871
75%,6.260000,-2.739928
max,7.730000,-1.270026


## 4. Coltheart's N (orthographic neighborhood size)

Coltheart's N = number of words in the English lexicon that differ from the target
by exactly **one letter substitution** at any position (same length required).

Used as a control by Michaelov et al. (2024) alongside word frequency.

We use the NLTK English words corpus (~236K words) as the reference lexicon.
Results are cached by unique critical word to avoid redundant computation.

In [7]:
nltk.download('words', quiet=True)
from nltk.corpus import words as nltk_words

# Group reference vocabulary by word length for fast lookup
vocab_by_len: dict[int, list[str]] = defaultdict(list)
for w in nltk_words.words():
    vocab_by_len[len(w)].append(w.lower())

total_vocab = sum(len(v) for v in vocab_by_len.values())
print(f"Reference vocabulary: {total_vocab:,} words across {len(vocab_by_len)} length bins")

Reference vocabulary: 236,736 words across 24 length bins


In [8]:
@lru_cache(maxsize=None)
def coltheart_n(word: str) -> int:
    """Count English words differing from `word` by exactly one letter substitution."""
    word = word.lower()
    n = len(word)
    candidates = vocab_by_len.get(n, [])
    return sum(
        1 for c in candidates
        if c != word and sum(a != b for a, b in zip(word, c)) == 1
    )

unique_words = stims['critical_word'].str.lower().unique()
print(f"Computing Coltheart's N for {len(unique_words)} unique critical words...")
stims['coltheart_n'] = stims['critical_word'].str.lower().apply(coltheart_n)

print(f"Cache size: {coltheart_n.cache_info().currsize} unique words computed")
stims[['critical_word', 'word_length', 'coltheart_n']].sample(8, random_state=42)

Computing Coltheart's N for 1320 unique critical words...
Cache size: 1320 unique words computed


,critical_word,word_length,coltheart_n
1539,she,3,18
2347,stipulation,11,1
2373,antidotes,9,0
1487,the,3,14
1097,drinking,8,0
2271,adoration,9,0
565,wall,4,28
1700,ideas,5,2


## 5. Save controls.csv

In [9]:
controls = stims[[
    'stim_id', 'dataset', 'critical_word',
    'word_length', 'word_position',
    'zipf_freq', 'log10_word_freq',
    'coltheart_n',
]].copy()

controls.to_csv(CONTROLS_OUT, index=False)
print(f"Saved {len(controls)} rows to {CONTROLS_OUT}")
controls.describe()

Saved 2808 rows to controls.csv


,stim_id,word_length,word_position,zipf_freq,log10_word_freq,coltheart_n
count,2808.000000,2808.000000,2808.000000,2808.000000,2808.000000,2808.000000
mean,1403.500000,5.415242,7.026353,5.004103,-3.995636,11.138533
std,810.744103,2.566662,3.908376,1.502235,1.501694,12.538322
min,0.000000,1.000000,0.000000,0.000000,-9.000000,0.000000
25%,701.750000,3.000000,4.000000,3.940000,-5.059932,1.000000
50%,1403.500000,5.000000,7.000000,4.920000,-4.079871,6.000000
75%,2105.250000,7.000000,9.000000,6.260000,-2.739928,17.000000
max,2807.000000,15.000000,21.000000,7.730000,-1.270026,57.000000


## 6. Merge demo

The controls file joins to any model output CSV (from `pipeline.ipynb`) on `stim_id`.
The model CSVs already contain `n_tokens` — no need to add it here.

### Recommended LME formula (for reference)
```
# Full model
N400_z ~ surprisal
       + zipf_freq + word_length + n_tokens + word_position + dataset
       + (1 + surprisal | subject)
       + (1 | stim_id)

# Null model (for LRT)
N400_z ~ zipf_freq + word_length + n_tokens + word_position + dataset
       + (1 + surprisal | subject)
       + (1 | stim_id)
```

In [10]:
import os

controls = pd.read_csv(CONTROLS_OUT)
erp      = pd.read_csv('combined_clean_n400.csv')

# Example: merge with a model output CSV
example_model_csv = os.path.join(OUTPUTS_DIR, 'gpt2_metrics.csv')
if os.path.exists(example_model_csv):
    model_df = pd.read_csv(example_model_csv)
    print(f"Model CSV columns: {list(model_df.columns)}")

    # Join model metrics + controls + ERP
    merged = (
        erp
        .merge(model_df[['stim_id', 'last_layer_surprisal', 'n_tokens_x',
                          'shallow_surprisal']].rename(columns={'n_tokens_x': 'n_tokens'}),
               on='stim_id')
        .merge(controls.drop(columns=['dataset', 'critical_word']), on='stim_id')
    )

    print(f"\nMerged shape: {merged.shape}")
    print(f"Columns: {list(merged.columns)}")
    merged.head(3)
else:
    # Fallback: just show controls + ERP merge
    merged = erp.merge(controls.drop(columns=['critical_word']), on=['stim_id', 'dataset'])
    print(f"ERP + controls shape: {merged.shape}")
    merged.head(3)

Model CSV columns: ['stim_id', 'dataset', 'critical_word', 'last_layer_surprisal', 'n_tokens_x', 'shallow_surprisal', 'n_tokens_y', 'layer_00', 'layer_01', 'layer_02', 'layer_03', 'layer_04', 'layer_05', 'layer_06', 'layer_07', 'layer_08', 'layer_09', 'layer_10', 'layer_11', 'lang_cosine_dist', 'lang_mae_update']

Merged shape: (45879, 17)
Columns: ['dataset', 'subject', 'stim_id', 'stim', 'stim_lower', 'condition', 'critical_word', 'meanAmp', 'meanAmp_z', 'last_layer_surprisal', 'n_tokens', 'shallow_surprisal', 'word_length', 'word_position', 'zipf_freq', 'log10_word_freq', 'coltheart_n']


In [11]:
# Quick correlation check: do controls correlate with N400 as expected?
# Higher frequency → less negative N400 (smaller absolute amplitude)
if 'meanAmp_z' in merged.columns:
    control_cols = ['zipf_freq', 'word_length', 'word_position', 'coltheart_n']
    corrs = merged[control_cols + ['meanAmp_z']].corr()['meanAmp_z'].drop('meanAmp_z')
    print("Pearson r with N400 (meanAmp_z):")
    print(corrs.to_string())
    print("\nNote: positive r = larger amplitude (more positive / less negative N400) with higher value")

Pearson r with N400 (meanAmp_z):
zipf_freq        0.018260
word_length      0.009105
word_position    0.052383
coltheart_n     -0.003890

Note: positive r = larger amplitude (more positive / less negative N400) with higher value
